# Engenharia de Prompts — notebook exploratório

Módulo 02 do BLIS. Este notebook é a versão interativa do `main.py`: serve para
**inspecionar prompts, rodar comparações parciais e olhar as saídas uma a uma**.

Artigos de base:

- Vaswani et al. (2017), *Attention Is All You Need* — o Transformer
- Brown et al. (2020), *Language Models are Few-Shot Learners* — zero/one/few-shot
- Wei et al. (2022), *Chain-of-Thought Prompting* — cadeia de raciocínio
- Zhou et al. (2023), *APE* — otimização automática da instrução

> As células marcadas **(sem custo)** não chamam a API. As marcadas **(custa)** chamam.

## 1. Preparação

In [ ]:
import sys, os

# O notebook precisa rodar a partir da pasta do módulo, onde estão os .py e data/
if not os.path.exists('prompts.py'):
    raise SystemExit('Rode este notebook a partir da pasta do módulo (onde está main.py).')

from config import SETTINGS, resumo_config
from dataset import carregar_perguntas, carregar_exemplos, exemplos_para, resumo, validar
from prompts import ESTRATEGIAS, PADRAO, obter, texto_do_prompt, tamanho_aproximado

perguntas = carregar_perguntas()
exemplos = carregar_exemplos()

print(resumo_config())
print(resumo(perguntas, exemplos))

### Validação do conjunto (sem custo)

O mais importante: nenhuma pergunta avaliada pode aparecer como demonstração
few-shot. Isso seria vazamento — o few-shot viraria consulta a gabarito.

In [ ]:
avisos = validar(perguntas, exemplos)
print('Nenhum problema.' if not avisos else '\n'.join(avisos))

## 2. Inspecionar os prompts (sem custo)

Antes de gastar qualquer token, vale ler o que efetivamente vai ser enviado.
A maior parte dos problemas de prompt engineering é visível aqui.

In [ ]:
pergunta = next(p for p in perguntas if p.id == 'arit_01')
print(f'{pergunta.id} ({pergunta.categoria}): {pergunta.pergunta}')
print(f'gabarito: {pergunta.esperado}')

In [ ]:
def mostrar(nome, pergunta, k=3):
    est = obter(nome)
    demos = exemplos_para(pergunta.categoria, exemplos, k) if est.usa_exemplos else []
    extra = {'k': k} if est.usa_exemplos else {}
    msgs = est.montar(pergunta.pergunta, demos, **extra)
    print('=' * 70)
    print(f'{nome}  (~{tamanho_aproximado(msgs)} tokens)  —  {est.artigo}')
    print('=' * 70)
    print(texto_do_prompt(msgs))
    print()

for nome in ['zero_shot', 'zero_shot_instrucao', 'few_shot', 'cot_few_shot']:
    mostrar(nome, pergunta)

### Custo relativo de cada estratégia (sem custo)

Few-shot e CoT few-shot gastam bem mais contexto. Um ganho de acurácia precisa
justificar esse custo — é o que a tabela de custo-benefício mede depois.

In [ ]:
tamanhos = []
for nome in PADRAO:
    est = obter(nome)
    demos = exemplos_para(pergunta.categoria, exemplos, SETTINGS.k_fewshot) if est.usa_exemplos else []
    extra = {'k': SETTINGS.k_fewshot} if est.usa_exemplos else {}
    tamanhos.append((nome, tamanho_aproximado(est.montar(pergunta.pergunta, demos, **extra))))

base = min(t for _, t in tamanhos)
for nome, t in sorted(tamanhos, key=lambda x: x[1]):
    print(f'{nome:<22} ~{t:>4} tokens   {t/base:>4.1f}x   ' + '#' * (t // 15))

## 3. Roteamento (sem custo)

O roteador classifica a pergunta e escolhe a estratégia. A heurística é grátis;
a confiança baixa é o sinal de que vale escalar para o classificador por LLM.

In [ ]:
from roteamento import Roteador, carregar_tabela, classificar_heuristico

roteador = Roteador(carregar_tabela())
print('Tabela ativa:')
print(roteador.resumo())

acertos = 0
print(f"\n{'id':<10} {'esperado':<14} {'obtido':<14} conf")
print('-' * 52)
for p in perguntas:
    cat, conf, _ = classificar_heuristico(p.pergunta)
    acertos += cat == p.categoria
    marca = '' if cat == p.categoria else '  <-- erro'
    print(f'{p.id:<10} {p.categoria:<14} {cat:<14} {conf:>4.0%}{marca}')
print('-' * 52)
print(f'acurácia do roteador: {acertos}/{len(perguntas)} = {acertos/len(perguntas):.1%}')

## 4. Comparar estratégias (**custa**)

A partir daqui é preciso ter `OPENROUTER_API_KEY` no `.env`.

Comece pequeno: poucas perguntas e poucas estratégias. Só depois rode a matriz inteira.

In [ ]:
from runner import executar
import relatorio

# Subconjunto pequeno para um primeiro teste barato
subconjunto = [p for p in perguntas if p.categoria in ('aritmetica', 'classificacao')][:6]
estrategias = ['zero_shot_instrucao', 'few_shot', 'cot_few_shot']

print(f'{len(subconjunto)} perguntas x {len(estrategias)} estratégias = '
      f'{len(subconjunto) * len(estrategias)} chamadas')

In [ ]:
execucao = executar(subconjunto, estrategias, exemplos, SETTINGS, verboso=True)

In [ ]:
relatorio.imprimir_geral(execucao)
relatorio.imprimir_por_categoria(execucao)
relatorio.imprimir_custo_beneficio(execucao)

### Onde as estratégias discordam

Estas são as perguntas que vale ler à mão: as que algumas estratégias acertam e
outras erram. É onde está a informação sobre *por que* uma técnica ajuda.

In [ ]:
relatorio.imprimir_divergencias(execucao)

### Ler uma saída inteira

A métrica diz *se* acertou. A saída bruta diz *como* o modelo chegou lá — e revela
quando a cadeia de raciocínio está fluente mas ilógica, o modo de falha que Wei et al.
observaram em modelos pequenos.

In [ ]:
for r in execucao.resultados:
    if r.estrategia == 'cot_few_shot':
        print('=' * 70)
        print(f'{r.pergunta_id} | esperado: {r.esperado} | '
              f"{'ACERTOU' if r.acertou else 'ERROU'}")
        print('=' * 70)
        print(r.saida)
        print()
        break

## 5. Variância entre execuções (**custa**)

Com `temperatura > 0`, a mesma estratégia dá respostas diferentes. Diferenças
pequenas entre estratégias podem ser só ruído — medir a variância evita concluir
demais de uma execução única.

In [ ]:
from dataclasses import replace

settings_variancia = replace(SETTINGS, temperature=0.7)
amostra = subconjunto[:3]

exec_var = executar(amostra, ['cot_zero_shot'], exemplos,
                    settings_variancia, repeticoes=3, verboso=False)

por_pergunta = {}
for r in exec_var.resultados:
    por_pergunta.setdefault(r.pergunta_id, []).append(r.acertou)

for pid, acertos in por_pergunta.items():
    estavel = 'estável' if len(set(acertos)) == 1 else 'INSTÁVEL'
    print(f'{pid}: {acertos}  ({estavel})')

## 6. APE — otimizar a instrução (**custa mais**)

Trata a instrução como um programa a ser otimizado: propõe candidatas a partir de
demonstrações, pontua cada uma pela acurácia de execução, fica com as melhores e
reamostra variações.

In [ ]:
import ape

resultado_ape = ape.buscar(
    perguntas, exemplos, SETTINGS,
    n_candidatas=4, rodadas=1, tamanho_subconjunto=6, verboso=True,
)
resultado_ape.imprimir()

In [ ]:
if resultado_ape.melhor:
    ape.salvar_instrucao(resultado_ape.melhor.instrucao)
    print('Instrução salva. Compare com as demais:')
    print('  python main.py comparar --estrategias ape few_shot cot_few_shot')

## 7. Aprender a tabela de roteamento

Fecha o ciclo: em vez de supor qual estratégia serve para cada tipo de pergunta,
deriva a tabela do que foi **medido**. O critério `custo` desempata pelo menor
gasto em tokens — se duas estratégias acertam o mesmo, não há razão para pagar mais.

In [ ]:
from roteamento import aprender_tabela, salvar_tabela

tabela = aprender_tabela(execucao, criterio='custo')
for categoria, estrategia in sorted(tabela.items()):
    print(f'{categoria:<16} -> {estrategia}')

# salvar_tabela(tabela)  # descomente para persistir

---

## Próximos passos

- Rode a matriz completa: `python main.py comparar`
- Exporte os relatórios: `python main.py relatorio --tudo`
- Troque `data/perguntas.json` pelas suas próprias perguntas

O roteiro completo com custos está em `REPRODUTIBILIDADE.md`.